# ARC Phase 2 — Full Arbitration Policy Training Pipeline

This notebook runs the complete training/evaluation workflow on a **single NVIDIA A100 80 GB**:

1. clone/update the ARC repository;
2. install the repository dependencies;
3. preflight the SFT / GRPO / validation / test datasets;
4. train the QLoRA SFT adapter in BF16;
5. train GRPO from the SFT adapter with the A100 baseline;
6. evaluate the final GRPO adapter on validation and test sets;
7. keep logs, checkpoints, prompt-budget diagnostics and effective training configuration.

**A100 baseline:** 4-bit NF4 + BF16, LoRA r=16/alpha=32/dropout=0.05 on q/k/v/o, GRPO G=8, prompt budget 4096, completion 256, batch 1, gradient accumulation 8, LR 1e-6, beta 0.01, paged AdamW 8-bit, gradient checkpointing, no vLLM.

## 1. Runtime Parameters

In [ ]:
from pathlib import Path
import os
import subprocess
import sys
import torch

REPO_URL = "https://github.com/beryl-07/arc.git"

cwd = Path.cwd().resolve()
if (cwd / ".git").exists() and (cwd / "src").exists():
    PROJECT_DIR = cwd
else:
    PROJECT_DIR = Path(os.getenv("ARC_REPO_DIR", "/workspace/arc")).resolve()

FORCE_RECLONE = False
RUN_SFT = True
RUN_GRPO = True
RUN_VALIDATION = True
RUN_TEST = True

# Single-A100 experiment: fail fast rather than silently changing hardware profiles.
EXPECTED_GPU_NAME = "A100"
EXPECTED_MIN_VRAM_GIB = 70.0
NUM_PROCESSES = 1
ACCELERATE_MIXED_PRECISION = "bf16"

# Explicit A100 GRPO baseline.
GRPO_NUM_GENERATIONS = 8
GRPO_MAX_STEPS = None  # None = use repository config (currently 50).
GRPO_MAX_PROMPT_LENGTH = 4096
GRPO_MAX_COMPLETION_LENGTH = 256
GRPO_PER_DEVICE_BATCH_SIZE = 1
GRPO_GRADIENT_ACCUMULATION_STEPS = 8
GRPO_LEARNING_RATE = 1e-6
GRPO_BETA = 0.01
GRPO_OPTIMIZER = "paged_adamw_8bit"

os.environ.setdefault("PYTORCH_ALLOC_CONF", "expandable_segments:True")
os.environ["GRPO_AUTO_MEMORY_SAFE_ON_SMALL_GPU"] = "false"

print("Project target:", PROJECT_DIR)
print("CUDA available:", torch.cuda.is_available())
print("GPU count detected:", torch.cuda.device_count())
print("Accelerate mixed precision:", ACCELERATE_MIXED_PRECISION)
print("A100 GRPO baseline: G=8, prompt=4096, completion=256, batch=1, accumulation=8")

## 2. Repository and Dependencies

In [ ]:
def run_cmd(cmd, *, cwd=None, env=None):
    print("$", " ".join(map(str, cmd)))
    return subprocess.run(cmd, cwd=cwd, env=env, check=True)

if subprocess.run(["bash", "-lc", "command -v git >/dev/null 2>&1"], check=False).returncode != 0:
    raise RuntimeError("git is required on the RunPod image but was not found.")

if FORCE_RECLONE and PROJECT_DIR.exists() and not (PROJECT_DIR == cwd and (PROJECT_DIR / ".git").exists()):
    import shutil
    shutil.rmtree(PROJECT_DIR)

if (PROJECT_DIR / ".git").exists():
    run_cmd(["git", "pull", "--ff-only"], cwd=PROJECT_DIR)
elif PROJECT_DIR.exists():
    raise RuntimeError(
        f"{PROJECT_DIR} exists but is not a git repository. "
        "Set ARC_REPO_DIR to a clean path or set FORCE_RECLONE=True."
    )
else:
    PROJECT_DIR.parent.mkdir(parents=True, exist_ok=True)
    run_cmd(["git", "clone", REPO_URL, str(PROJECT_DIR)])

os.environ.setdefault("SFT_VAL_PATH", "outputs/sft_val_messages.jsonl")
sys.path.insert(0, str(PROJECT_DIR))
os.environ["PYTHONPATH"] = str(PROJECT_DIR) + os.pathsep + os.environ.get("PYTHONPATH", "")

run_cmd([sys.executable, "-m", "pip", "install", "-q", "-r", str(PROJECT_DIR / "requirements.txt")])

from src.config import ARBITRATION_CONFIG

for module_name in ["torch", "transformers", "trl", "peft", "bitsandbytes", "datasets", "sklearn", "accelerate"]:
    module = __import__(module_name)
    print(f"{module_name}: {getattr(module, '__version__', 'unknown')}")

if not torch.cuda.is_available():
    raise RuntimeError("CUDA is unavailable. This notebook is intended for the A100 training pod.")
if torch.cuda.device_count() != 1:
    raise RuntimeError(f"Expected exactly 1 GPU for the A100 baseline; detected {torch.cuda.device_count()}.")

props = torch.cuda.get_device_properties(0)
vram_gib = props.total_memory / 1024**3
print(f"GPU: {props.name} | VRAM: {vram_gib:.1f} GiB | CUDA: {torch.version.cuda}")

if EXPECTED_GPU_NAME not in props.name or vram_gib < EXPECTED_MIN_VRAM_GIB:
    raise RuntimeError(
        f"This notebook is configured for an A100 80 GB. "
        f"Detected {props.name} with {vram_gib:.1f} GiB."
    )
if not torch.cuda.is_bf16_supported():
    raise RuntimeError("A100 baseline requires BF16, but this CUDA device does not report BF16 support.")

MODEL_NAME = ARBITRATION_CONFIG.model_name
SFT_TRAIN_PATH = PROJECT_DIR / ARBITRATION_CONFIG.sft_train_path
SFT_VAL_PATH = PROJECT_DIR / ARBITRATION_CONFIG.sft_val_path if ARBITRATION_CONFIG.sft_val_path else None
GRPO_TRAIN_PATH = PROJECT_DIR / ARBITRATION_CONFIG.grpo_train_path
VAL_PATH = PROJECT_DIR / ARBITRATION_CONFIG.val_path
TEST_PATH = PROJECT_DIR / ARBITRATION_CONFIG.test_path
SFT_OUTPUT_DIR = PROJECT_DIR / ARBITRATION_CONFIG.sft_output_dir
GRPO_OUTPUT_DIR = PROJECT_DIR / ARBITRATION_CONFIG.grpo_output_dir
OUTPUTS_DIR = PROJECT_DIR / ARBITRATION_CONFIG.outputs_dir
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

print("Model:", MODEL_NAME)
print("SFT train:", SFT_TRAIN_PATH)
print("GRPO train:", GRPO_TRAIN_PATH)
print("Validation:", VAL_PATH)
print("Test:", TEST_PATH)
print("SFT output:", SFT_OUTPUT_DIR)
print("GRPO output:", GRPO_OUTPUT_DIR)

## 3. Preflight Data Check

In [ ]:
import json
from src.arbitration_policy import build_arbitration_prompt

def build_sft_validation_file(source_path, output_path):
    output_path.parent.mkdir(parents=True, exist_ok=True)
    rows = []
    with source_path.open("r", encoding="utf-8") as handle:
        for line in handle:
            if not line.strip():
                continue
            record = json.loads(line)
            prompt_messages = build_arbitration_prompt(
                record, max_document_chars=0, max_total_document_chars=0
            )
            gold = record.get("gold_answer", [])
            response = gold if isinstance(gold, list) else [gold]
            assistant = {
                "role": "assistant",
                "content": json.dumps({
                    "strategy": "validation",
                    "response": response,
                    "justification": "Reference answer from the held-out validation split.",
                }, ensure_ascii=False),
            }
            rows.append({"messages": prompt_messages + [assistant]})
    with output_path.open("w", encoding="utf-8") as handle:
        for row in rows:
            handle.write(json.dumps(row, ensure_ascii=False) + "
")
    print(f"Built SFT validation file: {output_path} ({len(rows)} rows)")

if RUN_SFT and SFT_VAL_PATH is None:
    raise ValueError(
        "SFT_VAL_PATH is mandatory for SFT training and must point to "
        "a conversational JSONL file with `messages`."
    )
if RUN_SFT and SFT_VAL_PATH is not None and not SFT_VAL_PATH.exists():
    build_sft_validation_file(VAL_PATH, SFT_VAL_PATH)

paths = [SFT_TRAIN_PATH, GRPO_TRAIN_PATH, VAL_PATH, TEST_PATH]
if SFT_VAL_PATH is not None:
    paths.append(SFT_VAL_PATH)

def count_jsonl(path):
    with path.open("r", encoding="utf-8") as handle:
        return sum(1 for line in handle if line.strip())

for path in paths:
    if not path.exists():
        raise FileNotFoundError(path)
    print(f"{path.relative_to(PROJECT_DIR)}: {count_jsonl(path)} rows")

with SFT_TRAIN_PATH.open("r", encoding="utf-8") as handle:
    first_sft = json.loads(next(handle))
if "messages" not in first_sft:
    raise ValueError("SFT training file must contain conversational `messages` records.")

with GRPO_TRAIN_PATH.open("r", encoding="utf-8") as handle:
    first_grpo = json.loads(next(handle))
if "gold_answer" not in first_grpo:
    raise ValueError("GRPO training records must contain `gold_answer`.")

print("
A100 GRPO configuration")
print(json.dumps({
    "num_generations": GRPO_NUM_GENERATIONS,
    "max_steps": GRPO_MAX_STEPS if GRPO_MAX_STEPS is not None else ARBITRATION_CONFIG.grpo_max_steps,
    "max_prompt_length": GRPO_MAX_PROMPT_LENGTH,
    "max_completion_length": GRPO_MAX_COMPLETION_LENGTH,
    "per_device_train_batch_size": GRPO_PER_DEVICE_BATCH_SIZE,
    "gradient_accumulation_steps": GRPO_GRADIENT_ACCUMULATION_STEPS,
    "learning_rate": GRPO_LEARNING_RATE,
    "beta": GRPO_BETA,
    "optimizer": GRPO_OPTIMIZER,
    "bf16": True,
    "vllm": False,
}, indent=2))

## 4. SFT Training

In [ ]:
def run_checked(cmd, *, name=None):
    print("$", " ".join(map(str, cmd)))
    log_dir = OUTPUTS_DIR / "logs"
    log_dir.mkdir(parents=True, exist_ok=True)
    script_name = name or next(
        (Path(part).stem for part in cmd if str(part).endswith(".py")),
        Path(cmd[0]).stem,
    )
    log_path = log_dir / f"{script_name}.log"
    env = os.environ.copy()
    env["PYTHONUNBUFFERED"] = "1"

    process = subprocess.Popen(
        cmd, cwd=PROJECT_DIR, env=env, text=True,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=1,
    )
    assert process.stdout is not None
    with log_path.open("w", encoding="utf-8") as log_file:
        for line in process.stdout:
            print(line, end="")
            log_file.write(line)
            log_file.flush()

    return_code = process.wait()
    print(f"Log saved to: {log_path}")
    if return_code != 0:
        lines = log_path.read_text(encoding="utf-8", errors="replace").splitlines()
        tail = "
".join(lines[-100:])
        raise RuntimeError(
            f"Command failed with exit code {return_code}. Log: {log_path}
"
            f"--- Last log lines ---
{tail}"
        )
    return return_code

def accelerate_prefix():
    # Exactly one A100: no --multi_gpu and no automatic T4 fallback.
    return ["accelerate", "launch", "--num_processes", "1", "--mixed_precision", "bf16"]

if RUN_SFT:
    cmd = accelerate_prefix() + [
        str(PROJECT_DIR / "scripts" / "train_sft_arbitration.py"),
        "--model-name", MODEL_NAME,
        "--train-path", str(SFT_TRAIN_PATH),
        "--val-path", str(SFT_VAL_PATH),
        "--output-dir", str(SFT_OUTPUT_DIR),
        "--epochs", str(ARBITRATION_CONFIG.sft_num_epochs),
        "--learning-rate", str(ARBITRATION_CONFIG.sft_learning_rate),
        "--max-length", str(ARBITRATION_CONFIG.sft_max_seq_length),
        "--save-steps", str(ARBITRATION_CONFIG.sft_save_steps),
        "--logging-steps", str(ARBITRATION_CONFIG.sft_logging_steps),
        "--per-device-train-batch-size", str(ARBITRATION_CONFIG.sft_per_device_train_batch_size),
        "--gradient-accumulation-steps", str(ARBITRATION_CONFIG.sft_gradient_accumulation_steps),
        "--lora-r", str(ARBITRATION_CONFIG.lora_r),
        "--lora-alpha", str(ARBITRATION_CONFIG.lora_alpha),
        "--lora-dropout", str(ARBITRATION_CONFIG.lora_dropout),
        "--bf16",
        "--no-fp16",
    ]
    run_checked(cmd, name="train_sft_arbitration")
    if not (SFT_OUTPUT_DIR / "adapter_config.json").exists():
        raise FileNotFoundError(f"SFT adapter was not produced at {SFT_OUTPUT_DIR}")
else:
    print("Skipping SFT.")

## 5. GRPO Training

In [ ]:
if RUN_GRPO:
    if not (SFT_OUTPUT_DIR / "adapter_config.json").exists():
        raise FileNotFoundError(f"GRPO requires the SFT adapter at {SFT_OUTPUT_DIR}")

    cmd = accelerate_prefix() + [
        str(PROJECT_DIR / "scripts" / "train_grpo_arbitration.py"),
        "--sft-model-path", str(SFT_OUTPUT_DIR),
        "--train-path", str(GRPO_TRAIN_PATH),
        "--output-dir", str(GRPO_OUTPUT_DIR),
        "--num-generations", str(GRPO_NUM_GENERATIONS),
        "--max-prompt-length", str(GRPO_MAX_PROMPT_LENGTH),
        "--max-completion-length", str(GRPO_MAX_COMPLETION_LENGTH),
        "--per-device-train-batch-size", str(GRPO_PER_DEVICE_BATCH_SIZE),
        "--gradient-accumulation-steps", str(GRPO_GRADIENT_ACCUMULATION_STEPS),
        "--learning-rate", str(GRPO_LEARNING_RATE),
        "--beta", str(GRPO_BETA),
        "--optim", GRPO_OPTIMIZER,
        "--bf16",
        "--no-fp16",
        "--gradient-checkpointing",
    ]
    if GRPO_MAX_STEPS is not None:
        cmd += ["--max-steps", str(GRPO_MAX_STEPS)]

    run_checked(cmd, name="train_grpo_arbitration")
    if not (GRPO_OUTPUT_DIR / "adapter_config.json").exists():
        raise FileNotFoundError(f"GRPO adapter was not produced at {GRPO_OUTPUT_DIR}")
else:
    print("Skipping GRPO.")

## 6. Validation and Test

In [ ]:
def run_eval(split_name, data_path):
    output_path = OUTPUTS_DIR / f"arbitration_{split_name}_predictions.json"
    cmd = [
        sys.executable,
        str(PROJECT_DIR / "scripts" / "evaluate_arbitration_policy.py"),
        "--model-path", str(GRPO_OUTPUT_DIR),
        "--data-path", str(data_path),
        "--output-path", str(output_path),
        "--max-input-length", str(ARBITRATION_CONFIG.eval_max_input_length),
        "--max-new-tokens", str(ARBITRATION_CONFIG.eval_max_new_tokens),
        "--bf16",
        "--no-fp16",
    ]
    run_checked(cmd, name=f"evaluate_{split_name}")
    return output_path

if RUN_VALIDATION:
    if not (GRPO_OUTPUT_DIR / "adapter_config.json").exists():
        raise FileNotFoundError(f"Validation requires the GRPO adapter at {GRPO_OUTPUT_DIR}")
    run_eval("val", VAL_PATH)

if RUN_TEST:
    if not (GRPO_OUTPUT_DIR / "adapter_config.json").exists():
        raise FileNotFoundError(f"Test requires the GRPO adapter at {GRPO_OUTPUT_DIR}")
    run_eval("test", TEST_PATH)

print("
Pipeline complete.")
print("SFT adapter:", SFT_OUTPUT_DIR)
print("GRPO adapter:", GRPO_OUTPUT_DIR)
print("Logs:", OUTPUTS_DIR / "logs")
print("Evaluation outputs:", OUTPUTS_DIR)

## 7. RunPod operational notes

- Use a **single A100 80 GB** pod for the baseline experiment.
- The notebook fails fast if it detects the wrong GPU or more than one GPU.
- `RUN_SFT`, `RUN_GRPO`, `RUN_VALIDATION`, and `RUN_TEST` can be switched independently for resume/debug workflows.
- GRPO does **not** enable vLLM in this baseline.
- The GRPO script writes `prompt_budget_diagnostics.json` and `training_config.json`.
- Do not manually reduce the GRPO prompt budget to 512/384 on the A100: this run is intended to test the 4096-token baseline after fixing the T4 truncation problem.
- If you want a longer GRPO experiment later, change only `GRPO_MAX_STEPS` in this notebook.